# 01 - What is RAG and Why Does It Matter?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge at inference time. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents from a knowledge base and includes them as context in the prompt.

## The Problem RAG Solves

LLMs have three fundamental limitations that RAG addresses:

1. **Knowledge cutoff** — LLMs only know what was in their training data. They can't answer questions about recent events or proprietary information.
2. **Hallucination** — Without grounding, LLMs can generate plausible-sounding but incorrect information.
3. **Domain specificity** — General-purpose LLMs lack deep knowledge of specialized domains (your company's docs, legal texts, medical records, etc.).

## RAG Architecture Overview

```
┌─────────────┐    ┌──────────────┐    ┌────────────┐    ┌─────────────┐
│  Documents   │───>│   Chunking   │───>│ Embeddings │───>│ Vector Store│
│ (PDF,MD,CSV) │    │  & Splitting │    │   Model    │    │  (ChromaDB) │
└─────────────┘    └──────────────┘    └────────────┘    └──────┬──────┘
                                                                │
                                                                │ retrieve
                                                                │
┌─────────────┐    ┌──────────────┐    ┌────────────┐    ┌─────┴────────┐
│   Answer     │<──│     LLM      │<──│   Prompt    │<──│  Retriever   │
│              │    │(OpenAI/Claude)│   │ (question + │    │ (top-k docs) │
└─────────────┘    └──────────────┘    │  context)   │    └──────────────┘
                                       └────────────┘
```

The pipeline has two phases:
- **Indexing** (offline): Load documents → chunk → embed → store in vector DB
- **Querying** (online): User question → embed → retrieve similar chunks → inject into prompt → LLM generates answer

## What is LangChain?

**LangChain** is an open-source Python framework for building applications powered by LLMs. It's the most widely used framework in AI Engineering and provides building blocks for every stage of a RAG pipeline:

| Component | What it does | LangChain abstraction |
|-----------|-------------|----------------------|
| **Document Loaders** | Load PDFs, web pages, CSVs, databases into a unified format | `Document(page_content, metadata)` |
| **Text Splitters** | Break documents into retrieval-sized chunks | `RecursiveCharacterTextSplitter`, `TokenTextSplitter` |
| **Embeddings** | Convert text into dense vectors | `OpenAIEmbeddings`, `HuggingFaceEmbeddings` |
| **Vector Stores** | Store and search embeddings | `Chroma`, `FAISS`, `Pinecone` |
| **Retrievers** | Find relevant documents for a query | `VectorStoreRetriever`, `MultiQueryRetriever` |
| **Chat Models** | Generate responses (OpenAI, Anthropic, etc.) | `ChatOpenAI`, `ChatAnthropic` |
| **Chains (LCEL)** | Compose components with pipe (`\|`) syntax | `retriever \| prompt \| llm \| parser` |

### Why LangChain?

- **Unified interfaces**: All chat models implement `BaseChatModel`, all vector stores implement `VectorStore`. Swap providers without changing your pipeline code.
- **LCEL (LangChain Expression Language)**: Declarative, composable chains that are easy to read and modify.
- **Ecosystem**: 700+ integrations with data sources, LLMs, and tools.
- **Industry standard**: The most requested framework in AI Engineering job postings.

In this project, we use LangChain as the **glue layer** — our `src/rag_engine/` package wraps LangChain components with project-specific configuration. The notebooks teach what LangChain does at each step so you understand both the framework and the underlying concepts.

## Setup

Make sure you've installed the project and set up your `.env` file:

```bash
pip install -e ".[dev]"
cp .env.example .env
# Edit .env with your API keys
```

In [ ]:
import os
import sys

# Add project root to path so we can import rag_engine
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

## Demo: LLM Without Context (Hallucination)

Let's ask an LLM a question about a specific, niche topic without providing any context. This demonstrates the hallucination problem.

In [ ]:
from rag_engine.llm.provider import LLMProvider

llm = LLMProvider.get_llm(temperature=0.0)

# Ask about something specific that's in our sample data
question = "What are the four RAGAS evaluation metrics and what does each measure?"

response = llm.invoke(question)
print("Question:", question)
print("\nLLM Response (no context):")
print(response.content)

The LLM may give a reasonable answer here because RAGAS is in its training data. But for proprietary or recent data, it would hallucinate or refuse to answer.

## Demo: LLM With Manual Context (RAG Concept)

Now let's provide the relevant context manually — this is the core idea behind RAG.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

context = """
RAG systems should be evaluated on multiple dimensions:
- Faithfulness: Does the answer only contain information supported by the retrieved context?
- Answer relevancy: Is the answer actually relevant to the question asked?
- Context precision: Are the retrieved documents relevant to the question?
- Context recall: Are all the relevant documents in the knowledge base actually retrieved?
The RAGAS framework provides automated metrics for all of these dimensions.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based ONLY on the provided context. Context:\n{context}"),
    ("human", "{question}"),
])

chain = prompt | llm
response = chain.invoke({"context": context, "question": question})

print("Question:", question)
print("\nLLM Response (with context):")
print(response.content)

With context, the answer is **grounded in facts** — the model can only use what we provided. This is exactly what a RAG system automates:

1. The user asks a question
2. The system **retrieves** relevant documents from a knowledge base
3. The documents are **injected as context** into the prompt
4. The LLM **generates** an answer grounded in that context

## What's Next?

In the following notebooks, we'll build each component of this pipeline:

| Notebook | Topic |
|----------|-------|
| **02** | Document Loading — ingesting PDFs, Markdown, web pages, CSVs |
| **03** | Chunking Strategies — splitting documents into retrieval-sized pieces |
| **04** | Embeddings & Vector Stores — converting text to vectors and storing them |
| **05** | Retrieval Strategies — finding the right context for a query |
| **06** | Building the RAG Chain — wiring everything together with LangChain |
| **07** | Swappable LLM Providers — switching between OpenAI and Anthropic |
| **08** | Evaluation — measuring RAG quality with RAGAS |
| **09** | Advanced Techniques — HyDE, multi-query, re-ranking |

## Key References

- Lewis, P., et al. (2020). ["Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"](https://arxiv.org/abs/2005.11401)
- Gao, Y., et al. (2024). ["Retrieval-Augmented Generation for Large Language Models: A Survey"](https://arxiv.org/abs/2312.10997)
- [LangChain Documentation](https://python.langchain.com/docs/introduction/)